In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [2]:
df = pd.read_csv("people_wiki.csv")
df.head()

,URI,name,text
0,<http://dbpedia.org/resource/Digby_Morrell>,Digby Morrell,digby morrell born 10 october 1979 is a former...
1,<http://dbpedia.org/resource/Alfred_J._Lewy>,Alfred J. Lewy,alfred j lewy aka sandy lewy graduated from un...
2,<http://dbpedia.org/resource/Harpdog_Brown>,Harpdog Brown,harpdog brown is a singer and harmonica player...
3,<http://dbpedia.org/resource/Franz_Rottensteiner>,Franz Rottensteiner,franz rottensteiner born in waidmannsfeld lowe...
4,<http://dbpedia.org/resource/G-Enka>,G-Enka,henry krvits born 30 december 1974 in tallinn ...


### Explore data

In [3]:
obama = df[df['name'] == 'Barack Obama']
obama['text']

35817    barack hussein obama ii brk husen bm born augu...
Name: text, dtype: object

In [4]:
# Explore the entry for actor George Clooney
clooney = df[df['name'] == 'George Clooney']
clooney['text']

38514    george timothy clooney born may 6 1961 is an a...
Name: text, dtype: object

In [5]:
#Word counts for Obama acticle
from collections import Counter

obama_text = obama['text'].iloc[0]
obama_word_counts = Counter(obama_text.split())
obama_word_counts

Counter({'barack': 1,
         'hussein': 1,
         'obama': 9,
         'ii': 1,
         'brk': 1,
         'husen': 1,
         'bm': 1,
         'born': 2,
         'august': 1,
         '4': 1,
         '1961': 1,
         'is': 2,
         'the': 40,
         '44th': 1,
         'and': 21,
         'current': 1,
         'president': 4,
         'of': 18,
         'united': 3,
         'states': 3,
         'first': 3,
         'african': 1,
         'american': 3,
         'to': 14,
         'hold': 1,
         'office': 2,
         'in': 30,
         'honolulu': 1,
         'hawaii': 1,
         'a': 7,
         'graduate': 1,
         'columbia': 1,
         'university': 2,
         'harvard': 2,
         'law': 6,
         'school': 3,
         'where': 1,
         'he': 7,
         'served': 2,
         'as': 6,
         'review': 1,
         'was': 5,
         'community': 1,
         'organizer': 1,
         'chicago': 2,
         'before': 1,
         'earning': 1,
   

In [6]:
obama['word_count'] = obama['text'].str.split().str.len()
print(obama['word_count'])

35817    540
Name: word_count, dtype: int64


/var/folders/rq/17pdd6tn0q7cl934rk8rr1f40000gn/T/ipykernel_6025/4003385227.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  obama['word_count'] = obama['text'].str.split().str.len()


### Find most common words in Obama article

In [7]:
from collections import Counter

obama_text = obama['text'].iloc[0]
obama_word_counts = Counter(obama_text.split())

# Get the 10 most common words
most_common_words = obama_word_counts.most_common(10)
most_common_words


[('the', 40),
 ('in', 30),
 ('and', 21),
 ('of', 18),
 ('to', 14),
 ('his', 11),
 ('obama', 9),
 ('act', 8),
 ('a', 7),
 ('he', 7)]

In [8]:
#Create Word Counts
from sklearn.feature_extraction.text import CountVectorizer

count_vectorizer = CountVectorizer(stop_words='english')
word_count_matrix = count_vectorizer.fit_transform(df['text'])
word_count_matrix

<59071x548115 sparse matrix of type '<class 'numpy.int64'>'
	with 8078359 stored elements in Compressed Sparse Row format>

#### Compute TF-IDF for the entire corpus of articles

In [9]:
#Create TF-IDF Features
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf_vectorizer = TfidfVectorizer(stop_words='english')
tfidf_matrix = tfidf_vectorizer.fit_transform(df['text'])

tfidf_matrix

<59071x548115 sparse matrix of type '<class 'numpy.float64'>'
	with 8078359 stored elements in Compressed Sparse Row format>

#### Examine the TF-IDF for the Obama article

In [10]:
# Find TF-IDF for Barack Obama
obama_text = df[df['name'] == 'Barack Obama']['text'].values[0]
obama_vector = tfidf_vectorizer.transform([obama_text])

words = tfidf_vectorizer.get_feature_names_out()
scores = obama_vector.toarray().flatten()
obama_tfidf_df = pd.DataFrame({'word': words, 'tfidf': scores})
obama_tfidf_df = obama_tfidf_df[obama_tfidf_df['tfidf'] > 0].sort_values(by='tfidf', ascending=False)
obama_tfidf_df

,word,tfidf
358378,obama,0.413495
45070,act,0.282170
259085,iraq,0.171970
292368,law,0.163903
138523,control,0.149369
...,...,...
23034,2008,0.019848
22237,2007,0.019679
534477,won,0.018853
541245,years,0.016415


In [11]:
# use cosine similarity to find people with similar writing style/content
from sklearn.metrics.pairwise import cosine_similarity

# Compare Barack Obama to all others
obama_index = df[df['name'] == 'Barack Obama'].index[0]
similarities = cosine_similarity(tfidf_matrix[obama_index], tfidf_matrix).flatten()

# Add similarity column
df['similarity_to_obama'] = similarities
most_similar = df.sort_values(by='similarity_to_obama', ascending=False).head(10)
most_similar

,URI,name,text,similarity_to_obama
35817,<http://dbpedia.org/resource/Barack_Obama>,Barack Obama,barack hussein obama ii brk husen bm born augu...,1.000000
24478,<http://dbpedia.org/resource/Joe_Biden>,Joe Biden,joseph robinette joe biden jr dosf rbnt badn b...,0.321219
38376,<http://dbpedia.org/resource/Samantha_Power>,Samantha Power,samantha power born september 21 1970 is an ir...,0.271129
57108,<http://dbpedia.org/resource/Hillary_Rodham_Cl...,Hillary Rodham Clinton,hillary diane rodham clinton hlri dan rdm klnt...,0.256239
38714,<http://dbpedia.org/resource/Eric_Stern_(polit...,Eric Stern (politician),eric stern is the director of operations for t...,0.252736
46140,<http://dbpedia.org/resource/Robert_Gibbs>,Robert Gibbs,robert lane gibbs born march 29 1971 is an ame...,0.235931
18827,<http://dbpedia.org/resource/Henry_Waxman>,Henry Waxman,henry arnold waxman born september 12 1939 is ...,0.227405
44681,<http://dbpedia.org/resource/Jesse_Lee_(politi...,Jesse Lee (politician),jesse lee born 1979 was named the white house ...,0.225401
6796,<http://dbpedia.org/resource/Eric_Holder>,Eric Holder,eric himpton holder jr born january 21 1951 is...,0.220879
2412,<http://dbpedia.org/resource/Joe_the_Plumber>,Joe the Plumber,samuel joseph wurzelbacher wrzlbkr born decemb...,0.216740


In [12]:
df

,URI,name,text,similarity_to_obama
0,<http://dbpedia.org/resource/Digby_Morrell>,Digby Morrell,digby morrell born 10 october 1979 is a former...,0.008082
1,<http://dbpedia.org/resource/Alfred_J._Lewy>,Alfred J. Lewy,alfred j lewy aka sandy lewy graduated from un...,0.012457
2,<http://dbpedia.org/resource/Harpdog_Brown>,Harpdog Brown,harpdog brown is a singer and harmonica player...,0.013357
3,<http://dbpedia.org/resource/Franz_Rottensteiner>,Franz Rottensteiner,franz rottensteiner born in waidmannsfeld lowe...,0.014082
4,<http://dbpedia.org/resource/G-Enka>,G-Enka,henry krvits born 30 december 1974 in tallinn ...,0.004633
...,...,...,...,...
59066,<http://dbpedia.org/resource/Olari_Elts>,Olari Elts,olari elts born april 27 1971 in tallinn eston...,0.006595
59067,<http://dbpedia.org/resource/Scott_F._Crago>,Scott F. Crago,scott francis crago born july 26 1963 twin bro...,0.006749
59068,<http://dbpedia.org/resource/David_Cass_(footb...,David Cass (footballer),david william royce cass born 27 march 1962 in...,0.011082
59069,<http://dbpedia.org/resource/Keith_Elias>,Keith Elias,keith hector elias born february 3 1972 in lac...,0.025106
